# Levadura: dos destinos y una respuesta que se adapta

Compara un modelo docente de autoactivación con una analogía local de control. Los estados y parámetros de ambas simulaciones se definen por separado; no son ajustes a los artículos.

Lecciones: 22 · Casos de levadura: autoactivación, adaptación y programa explicado, 11 · Control integral biomolecular: adaptación robusta y sus límites.

Fuentes: [A01](https://www.math.uwaterloo.ca/~bingalls/MMSB/), [A02](https://www.mat.univie.ac.at/~gerald/ftp/book-ode/), [A25](https://pmc.ncbi.nlm.nih.gov/articles/PMC125456/), [A26](https://pmc.ncbi.nlm.nih.gov/articles/PMC3109981/), [R05](https://docs.scipy.org/doc/scipy/reference/generated/scipy.integrate.solve_ivp.html).

## Lectura del código

La primera parte encuentra raíces de una autoactivación y usa el signo de la derivada para clasificarlas. La segunda usa otros estados y otra pregunta: h es desviación de respuesta y g compensación acumulada. La fórmula analítica permite comprobar tiempo, signos y orden de ambas salidas. La última parte añade fuga, calcula el nuevo equilibrio y evalúa las dos pendientes allí. Las simulaciones son analogías docentes separadas de los datos de los dos artículos motivadores.

## Preparación
El primer bloque localiza los módulos del paquete. `__file__` existe al ejecutar un script; en Jupyter se utiliza la carpeta actual. `sys.path.insert` permite importar el módulo de modelos. Ejecuta desde `python/cuadernos/` o desde `python/`. Lee los comentarios antes de cada llamada.

In [ ]:
# Localizar los módulos en python/, tanto como script como en Jupyter.
from pathlib import Path
import sys
base = Path(__file__).resolve().parent.parent if '__file__' in globals() else Path.cwd()
if not (base / 'circuitos_casos.py').exists():
    base = base.parent
sys.path.insert(0, str(base))
# NumPy organiza vectores; SciPy integra; Matplotlib dibuja resultados.
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
import circuitos_casos as cc


## Encontrar y clasificar equilibrios, sin inferirlos a ojo
Antes de ejecutar, identifica qué entra, qué sale y las unidades. El capítulo correspondiente deriva las fórmulas que se están comprobando.

Fuentes: [A01](https://www.math.uwaterloo.ca/~bingalls/MMSB/), [A02](https://www.mat.univie.ac.at/~gerald/ftp/book-ode/), [A25](https://pmc.ncbi.nlm.nih.gov/articles/PMC125456/), [A26](https://pmc.ncbi.nlm.nih.gov/articles/PMC3109981/), [R05](https://docs.scipy.org/doc/scipy/reference/generated/scipy.integrate.solve_ivp.html).

In [ ]:
raices = cc.raices_autoactivacion(a=3, basal=.02)
pendientes = cc.pendiente_autoactivacion(raices, a=3)
print('Equilibrios:', raices)
print('Derivadas locales:', pendientes)
np.testing.assert_allclose(cc.autoactivacion(0,raices,3,.02),0,atol=1e-12)
t = np.linspace(0, 25, 501)
for x0 in [.1, .8]:
    sol = solve_ivp(cc.autoactivacion, (0,25), [x0], t_eval=t,
                   args=(3,.02,2), rtol=1e-9, atol=1e-11)
    if not sol.success:
        raise RuntimeError(sol.message)
    plt.plot(t, sol.y[0], label=f'x inicial={x0}')
plt.xlabel('Tiempo normalizado'); plt.ylabel('Activador normalizado')
plt.legend(); plt.show()


## Comprobar la analogía de adaptación con una solución exacta
Antes de ejecutar, identifica qué entra, qué sale y las unidades. El capítulo correspondiente deriva las fórmulas que se están comprobando.

Fuentes: [A01](https://www.math.uwaterloo.ca/~bingalls/MMSB/), [A02](https://www.mat.univie.ac.at/~gerald/ftp/book-ode/), [A25](https://pmc.ncbi.nlm.nih.gov/articles/PMC125456/), [A26](https://pmc.ncbi.nlm.nih.gov/articles/PMC3109981/), [R05](https://docs.scipy.org/doc/scipy/reference/generated/scipy.integrate.solve_ivp.html).

In [ ]:
t = np.linspace(0, 20, 401)
sol = solve_ivp(cc.adaptacion, (0,20), [0,0], t_eval=t,
                rtol=1e-10, atol=1e-12)
if not sol.success:
    raise RuntimeError(sol.message)
exacta = cc.adaptacion_exacta(t)
np.testing.assert_allclose(sol.y, exacta, rtol=2e-8, atol=2e-10)
plt.plot(t, sol.y[0], label='Desviación h')
plt.plot(t, sol.y[1], label='Compensación g')
plt.plot(t, exacta[0], '--', label='h analítica')
plt.xlabel('Tiempo docente'); plt.ylabel('Estado en escala normalizada')
plt.legend(); plt.show()


## Añadir fuga y comprobar qué igualdad deja de cumplirse
Antes de ejecutar, identifica qué entra, qué sale y las unidades. El capítulo correspondiente deriva las fórmulas que se están comprobando.

Fuentes: [A01](https://www.math.uwaterloo.ca/~bingalls/MMSB/), [A02](https://www.mat.univie.ac.at/~gerald/ftp/book-ode/), [A25](https://pmc.ncbi.nlm.nih.gov/articles/PMC125456/), [A26](https://pmc.ncbi.nlm.nih.gov/articles/PMC3109981/), [R05](https://docs.scipy.org/doc/scipy/reference/generated/scipy.integrate.solve_ivp.html).

In [ ]:
for fuga in [0, .1, .3]:
    equilibrio = cc.equilibrio_adaptacion(fuga=fuga)
    # Evaluar ambos balances debe devolver pendientes cero.
    np.testing.assert_allclose(cc.adaptacion(0,equilibrio,fuga=fuga),0,atol=1e-12)
    print('Fuga, respuesta residual, compensación:', fuga, *equilibrio)


## Comprueba con tus palabras
Deriva el polinomio lambda^2+b*lambda+a*c y comprueba sus autovalores. Explica por qué h puede ser negativa en una analogía de desviaciones, mientras que la concentración del activador del primer modelo debe permanecer no negativa. No confundas bimodalidad observada y dos raíces estables deterministas.

Los `assert_allclose` comparan consecuencias analíticas o implementaciones equivalentes; no validan experimentalmente las hipótesis.